In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import shutil
import torch
import numpy as np
import pretty_midi as pyd
import demucs.separate

from musecoco.hf_musecoco.midi_utils.utils_midi import RemiTokenizer
from transformers import MusicgenForConditionalGeneration, MusicgenConfig
from model_blip2 import Blip2Musecoco
from dataset import collate_fn_inference, TK
from tqdm import tqdm

from utils.inference import read_leadsheet, read_audio_to_codec, get_chord_change_steps, add_pedal


os.environ['CUDA_VISIBLE_DEVICES']= '6'
DEVICE = 'cuda:0'



def audio_voice_separate(source_path):
    demucs.separate.main(["--mp3", "--two-stems", "vocals", "-n", "mdx_extra", source_path])
    save_path = source_path.replace('.mp3', '') + '_accompaniment.mp3'

    tmp_dir = f"separated/mdx_extra/{source_path.split('/')[-1].replace('.mp3', '')}"
    shutil.move(os.path.join(tmp_dir, "no_vocals.mp3"), save_path)
    shutil.rmtree(tmp_dir)


In [ ]:
# load audio encodec
config=MusicgenConfig.from_pretrained("facebook/musicgen-small")
config.decoder.return_dict_in_generate = True
config.decoder.output_hidden_states = True
auditory_encoder = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small", config=config)
audio_encoder = auditory_encoder.get_audio_encoder()
audio_encoder.to(DEVICE)

# load blip2musecoco model
model = Blip2Musecoco()
state_dict = torch.load("/data2/zhaojw/blip2_a2s_musecoco_PIAST_POP909_LoRA_002_epoch.pt", map_location='cpu')
for key in list(state_dict.keys()):
            new_key = key.replace('blip2qformer.ln_audio', 'blip2qformer.layer_norm')
            state_dict[new_key] = state_dict.pop(key)  
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval();

In [ ]:
# set audio and leadsheet path
audio_path = "/data2/zhaojw/LAVIS/inference_style_transfer/audio/ragtime.mp3"
midi_path = "/data2/zhaojw/LAVIS/inference_style_transfer/leadsheet_processed/867-4.mid"
audio_start_time = 11
midi_start_bar = int(midi_path.split('-')[-1].split('.')[0])
num_sample = 4
save_path = '/data2/zhaojw/a2s_style_learning'

# separate audio to accompaniment, if its not already accompaniment
#if os.path.exists(audio_path.replace('.mp3', '_accompaniment.mp3')):
#    print("Accompaniment file already exists.")
#else:
#    audio_voice_separate(audio_path)

#process audio from waveform to codec
codec = read_audio_to_codec(audio_path, audio_encoder, device=DEVICE, offset=audio_start_time).detach().cpu().numpy()
print('codec', codec.shape)

output_values = audio_encoder.decode(
    torch.from_numpy(codec).to(DEVICE).unsqueeze(0).unsqueeze(0),
    audio_scales=[None]*1)
import soundfile as sf
sf.write(os.path.join(save_path, 'audio_recon.wav'), output_values.audio_values[0][0].detach().cpu().numpy(), 32000, 'PCM_24')

# read midi leadsheet into octMIDI format
downbeat_time, leadsheet_events, leadsheet_times = read_leadsheet(midi_path)
#extropolate the last downbeat time
downbeat_time = np.append(downbeat_time, downbeat_time[-1] + (downbeat_time[-1] - downbeat_time[-2]))   


#reconstruct to midi
#ls = encoding_to_MIDI(leadsheet_events)
#ls.dump(os.path.join(save_path, 'leadsheet_recon.mid'))

# process leadsheet into REMI format
STAR_BAR = midi_start_bar
DURATION = 16
HOP_LEN = 2
BAR_LEN = 4
BATCH = [codec, downbeat_time, None, None, leadsheet_times, leadsheet_events]
audio_slices, audio_mask, ls_slices = collate_fn_inference(BATCH, DEVICE, [STAR_BAR, STAR_BAR+DURATION], HOP_LEN, BAR_LEN)
print(audio_slices.shape, audio_mask.shape, [ls_slices[idx].shape for idx in range(len(ls_slices))])

# get chord change steps in order to add pedal in a rule-based manner 
chord_change_steps = get_chord_change_steps(leadsheet_events, leadsheet_times, downbeat_time, STAR_BAR, STAR_BAR+DURATION)
print(chord_change_steps)

codec (4, 11408)
torch.Size([7, 4, 508]) torch.Size([7, 512]) [torch.Size([1, 260]), torch.Size([1, 260]), torch.Size([1, 278]), torch.Size([1, 273]), torch.Size([1, 250]), torch.Size([1, 250]), torch.Size([1, 241])]
[0, 3, 4, 6, 8, 9, 12, 15, 16, 18, 20, 21, 24, 27, 28, 30, 32, 33, 36, 39, 40, 42, 44, 45, 48, 51, 52, 54, 56, 57, 60, 64, 68, 72, 76]


In [7]:
for num in range(1):
    #try:
    generation_slices = []  #collected generated slices
    intervals = []  #collected intervals of each generation slice
    for idx in tqdm(range(len(audio_slices))):
        decoder_input_ids = ls_slices[idx]  #get lead sheet control tokens
        if idx > 0:
            #get bar 3~4 of last generation as bar 1~2 of current generation
            enter_point = torch.nonzero(generation_slices[-1]==5)[HOP_LEN-1, 1] + 1 # 5 is the new_bar token.
            try:
                break_point = torch.nonzero(generation_slices[-1]==5)[BAR_LEN-1, 1] + 1
            except IndexError:
                break_point = -1 
            decoder_input_ids = torch.cat([decoder_input_ids, generation_slices[-1][:, enter_point: break_point]], dim=1)
            intervals.append(enter_point)
        
        token_pred = model.generate_nucleus2(audio_slices[idx: idx+1], decoder_input_ids, audio_mask[idx: idx+1], t=1, p=None, k=15)

        enter_point = torch.nonzero(token_pred==984)[-1, 1] + 1 # 984 is the sos token
        token_pred = token_pred[:, enter_point:]    #discard attribute and lead sheet control tokens

        generation_slices.append(token_pred)

    generation_reult = torch.cat([generation_slices[idx][:, :itv] \
                            for idx, itv in enumerate(intervals)] \
                                + [generation_slices[-1]], 
                                dim=1)
   
    # convert generated tokens to midi
    pred_sample = [TK._convert_id_to_token(tk) for tk in generation_reult[0].detach().cpu().numpy()]
    pred_sample = [tk for tk in pred_sample if ('-' in tk)]
    midi_tok = RemiTokenizer()
    pred_sample = midi_tok.remi_to_midi(pred_sample, ignore_velocity=False)
    pred_sample.dump(os.path.join(save_path, f'{str(num).zfill(2)}.mid'))

    # add pedal to the piano cover
    pred_sample = pyd.PrettyMIDI(os.path.join(save_path, f'{str(num).zfill(2)}.mid'))
    pred_sample = add_pedal(pred_sample, chord_change_steps)
    pred_sample.write(os.path.join(save_path, f'{str(num).zfill(2)}.mid'))
    #except Exception as e:
    #    print(f"Error occurred during generation: {e}")
    #    continue

100%|██████████| 7/7 [00:41<00:00,  5.87s/it]
